# 00 — Profile / Decision-package config check

**Profile:** `workflow_update` + provisional override `workflow_update_downstream`  
**Status:** `NON_BASELINE_RUN` (chưa đủ data_gate + scenario_gate + product_gate)

Notebook này chỉ kiểm `Config.load_profiled` + `workflow_runtime()` + khóa Decision-package.
Không chạy pipeline nặng.


In [ ]:
from __future__ import annotations

import time
from pathlib import Path

from qshield_contracts.config import Config
from qshield_contracts.schemas.downstream import (
    validate_transaction_cost_excludes_liquidity,
)

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "configs" / "base.yaml").exists():
    PROJECT_ROOT = Path.cwd().resolve().parents[1]

BASE = PROJECT_ROOT / "configs" / "base.yaml"
PROFILE = PROJECT_ROOT / "configs" / "profiles" / "workflow_update.yaml"
OVERRIDE = PROJECT_ROOT / "configs" / "provisional" / "workflow_update_downstream.yaml"
print("PROJECT_ROOT", PROJECT_ROOT)
print("NON_BASELINE_RUN — Decision-package provisional path")

In [ ]:
t0 = time.perf_counter()
cfg = Config.load_profiled(BASE, PROFILE, OVERRIDE)
load_s = time.perf_counter() - t0

t1 = time.perf_counter()
runtime = cfg.workflow_runtime()
runtime_s = time.perf_counter() - t1

print(f"load_profiled: {load_s:.4f}s")
print(f"workflow_runtime: {runtime_s:.4f}s")
print(
    {
        "profile_id": runtime.profile_id,
        "profile_status": runtime.profile_status,
        "candidate_count": runtime.candidate_count,
        "total_decision_bits": runtime.total_decision_bits,
        "structured_sample_count": runtime.structured_sample_count,
    }
)

In [ ]:
checks = {
    "fee": cfg["transaction_cost"]["fee"] == 0.0015,
    "spread": cfg["transaction_cost"]["spread"] == 0.0010,
    "liquidity_penalty": cfg["transaction_cost"]["liquidity_penalty"] == 0.0005,
    "weight_sum_tolerance": cfg["weight_sum_tolerance"] == 1e-8,
    "cash_increment": cfg["target_cash_increment"] == 0.10,
    "max_reduction": cfg["maximum_reduction"] == 0.30,
    "cvar_first": cfg["financial_objective"]["priority"] == "cvar_first",
    "rerank_top20": cfg["reranking"]["top_distinct_feasible"] == 20,
    "polish_pp": cfg["local_polishing"]["max_adjustment_pp"] == 5,
    "materiality_1pct": cfg["materiality"]["true_cvar_relative_reduction_min"] == 0.01,
    "qaoa_seeds_10": len(cfg["quantum"]["qaoa"]["seeds"]) == 10,
    "warm_start": cfg["quantum"]["qaoa"]["warm_start"] is True,
    "non_final_false": cfg["quantum"]["qaoa"]["NON_FINAL_CONFIG"] is False,
    "cost_sensitivity": "cost_sensitivity" in cfg,
    "performance_budget": "performance_budget" in cfg,
    "surrogate_provisional": cfg["surrogate_validation"]["status"]
    == "PROVISIONAL_PENDING_OWNER",
    "benchmark_same_hash": cfg["benchmark"]["require_same_qubo_hash"] is True,
    "profile_status_non_baseline": runtime.profile_status == "NON_BASELINE_RUN",
}

validate_transaction_cost_excludes_liquidity(cfg)
failed = [k for k, ok in checks.items() if not ok]
print("checks", checks)
print("failed", failed)
assert not failed, failed
print("OK — Decision-package keys present after load_profiled")
print(f"total_elapsed_s={load_s + runtime_s:.4f}")